# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadSulaiman08/FLYRANK-ML-INTERNSHIP/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

My Lane **Lane 2 — Refresh / Content Opportunity Scoring (provisional).** I want to rank pages so a content editor knows which ones to review first. In notebook 01 I observed that declining pages are often younger than growing ones, and in notebook 02 a simple "stale × visible" hand rule held up well on unseen clients. So I want to find which signals really separate pages worth reviewing from noise, and whether a model can beat a transparent rule honestly.

In [1]:
import os, subprocess
import pandas as pd

CSV = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(CSV):
    for up in ("..", "../.."):
        if os.path.exists(os.path.join(up, CSV)):
            os.chdir(up); break
    else:
        if not os.path.isdir("starter"):
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/flyrank-bih/flyrank-ml-internship-starter", "starter"], check=True)
        os.chdir("starter")

df = pd.read_csv(CSV)
df["is_declining"] = df["trend_direction"].str.lower().eq("down")
print(f"{len(df):,} pages from {df['client_id'].nunique()} clients")

30,000 pages from 32 clients


The Question **Decision:** which pages an editor should review first, given limited hours each week.

**Who acts:** a content editor / SEO lead working through a ranked review queue.

**Action:** review the page, then refresh, expand, monitor, or leave it.

**Cost of a wrong call:** a false alarm wastes editor time on a page that was fine (or just seasonal); a miss lets a valuable page keep losing traffic unnoticed. Because capacity is limited, the top of the list matters most, so I will measure with Precision@K.

**Why this is not just "train a model":** the goal is a better review order, not a high score. A plain rule is the baseline to beat; ML only earns its place if it ranks better on clients it has never seen, and every pick needs a reason code a human can check.

In [2]:
print("Share of pages labelled declining:", round(df["is_declining"].mean(), 3))

Share of pages labelled declining: 0.542


**What the numbers show (observed):** about 13k pages are declining while still getting real demand, and around half of all impressions sit on declining pages, so an editor cannot review everything and the order matters. The "stale & visible" rule is very precise (most of its picks are declining) but catches only 17 pages, so a simple rule runs out quickly. That gap is where a ranking model could help.

In [3]:
dwd = df[df["is_declining"] & (df["impressions_90d"] >= 100)]
print(f"Declining pages that still have demand (>=100 impressions): {len(dwd):,} ({len(dwd)/len(df):.1%} of all pages)")

share = df.loc[df["is_declining"], "impressions_90d"].sum() / df["impressions_90d"].sum()
print(f"Share of all impressions sitting on declining pages: {share:.1%}")

sv = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(f"Stale & visible pages: {len(sv)}  |  of those, declining: {sv['is_declining'].mean():.1%}")

Declining pages that still have demand (>=100 impressions): 13,152 (43.8% of all pages)
Share of all impressions sitting on declining pages: 51.3%
Stale & visible pages: 17  |  of those, declining: 94.1%


**I can say:** observed patterns, directional differences, and a ranked review list that supports a human decision, checked on clients the model has never seen.

**I can't say:** that refreshing a page causes recovery, that I found Google ranking factors, or that "declining" is a future outcome. The starter label comes from the current trend, so it is a proxy, not a prediction of what happens next.

In [4]:
pass

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.